# Ringkasan Pengerjaan

Pipeline pembuatan dataset v2, mengatasi bias ukuran graf yang ditemukan di notebook eksplorasi awal.

**Isi**: pengumpulan ulang kontrak normal dengan distribusi ukuran diseimbangkan terhadap kelas reentrancy, tiga iterasi perbaikan resolver versi Solidity, validasi kompilasi, ekstraksi CFG (normal dan reentrancy), pruning, dan konversi akhir ke `v2_cfg_prune.pt`.

**Hasil akhir**: 610 graf latih (366 normal, 244 reentrancy) dan 16 graf benchmark pasangan CEI, dimensi fitur 22.

**Lanjutan**: dataset ini dipakai untuk training model di `PROJECT_v2_model_dan_shap.ipynb` dan eksperimen fitur posisi di `PROJECT_v2_fitur_posisi.ipynb`.

# Setup dan Persiapan

In [ ]:
# === SEL 1: SETUP ===
from google.colab import drive
drive.mount('/content/drive')

!pip install -q slither-analyzer solc-select datasets

import os, json, re, math, shutil, subprocess, time
import numpy as np, pandas as pd

D = '/content/drive/MyDrive/PROJECT'
BASE = '/content/data/raw/dataset_v2'
OUT_CFG = f'{D}/data/processed/v2_cfg'
for p in [f'{BASE}/0_normal', f'{BASE}/1_reentrancy', OUT_CFG]:
    os.makedirs(p, exist_ok=True)

# salin binary solc dari Drive ke lokal lalu beri izin eksekusi
if not os.path.exists(os.path.expanduser('~/.solc-select/artifacts')):
    os.makedirs(os.path.expanduser('~/.solc-select'), exist_ok=True)
    !cp -r {D}/solc_binaries/artifacts ~/.solc-select/
    !chmod -R +x ~/.solc-select/artifacts/
!solc-select versions | head -3

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.0/765.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.3/586.3 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.4/460.4 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# === SEL 2: SALIN 244 KONTRAK REENTRANCY YANG SUDAH VALID ===
if not os.path.exists('/content/data/raw/dataset_penelitian'):
    os.makedirs('/content/data/raw', exist_ok=True)
    !unzip -q -o {D}/data/raw/dataset_final.zip -d /content/data/raw/

src = '/content/data/raw/dataset_penelitian/1_reentrancy'
for f in os.listdir(src):
    if f.endswith('.sol'):
        shutil.copy(os.path.join(src, f), f'{BASE}/1_reentrancy/{f}')

ren = pd.read_csv(f'{D}/data/processed/ukuran_reentrancy.csv')
print(f'{len(os.listdir(f"{BASE}/1_reentrancy"))} kontrak reentrancy disalin')
print(f'acuan ukuran: {len(ren)} baris')

244 kontrak reentrancy disalin
acuan ukuran: 244 baris


In [ ]:
# === SEL 3: FUNGSI BANTU ===
from packaging.specifiers import SpecifierSet
from packaging.version import Version
from solc_select import solc_select

terpasang = sorted([Version(v) for v in solc_select.installed_versions()])

def get_pragma(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        c = f.read()
    m = re.search(r'pragma\s+solidity\s*([^;]+);', c)
    return m.group(1).strip() if m else 'TIDAK_DITEMUKAN'

def resolve_solc(pragma_str):
    if not pragma_str or str(pragma_str) == 'TIDAK_DITEMUKAN':
        return None
    s = str(pragma_str).replace('^', '~=').strip()
    try:
        spec = SpecifierSet(','.join(p.strip() for p in s.split() if p.strip()))
        cocok = [v for v in terpasang if v in spec]
        if cocok:
            return str(max(cocok))
    except Exception:
        pass
    m = re.search(r'(\d+\.\d+\.\d+)', str(pragma_str))
    return m.group(1) if m else None

def kontrak_utuh(sc):
    if not sc or len(sc) < 200: return False
    if sc.count('{') != sc.count('}'): return False
    if sc.count('(') != sc.count(')'): return False
    if sc.count('"') % 2 != 0: return False
    if 'pragma solidity' not in sc: return False
    return True

print('fungsi siap, versi solc terbaca:', len(terpasang))

fungsi siap, versi solc terbaca: 61


# Pengumpulan Kontrak Normal Berimbang Ukuran

In [ ]:
import os, pandas as pd
MANIFEST = f'{D}/data/processed/v2_manifest.csv'
BATAS = [0, 12389, 18004, 40626, 69888, 240707, 10**12]
TARGET = [50, 75, 125, 125, 75, 50]

def kelompok(n):
    for i in range(len(BATAS) - 1):
        if BATAS[i] <= n < BATAS[i + 1]:
            return i
    return len(TARGET) - 1

mf = pd.read_csv(MANIFEST)
print(f'{len(mf)} baris manifest, OK={sum(mf.status=="OK")}, GAGAL={sum(mf.status=="GAGAL")}')
print('file .sol normal:', len(os.listdir(f'{BASE}/0_normal')))
print('file .sol reentrancy:', len(os.listdir(f'{BASE}/1_reentrancy')))

500 baris manifest, OK=374, GAGAL=126
file .sol normal: 0
file .sol reentrancy: 0


In [ ]:
# === SEL 4: UNDUH KONTRAK NORMAL BERIMBANG ===
from datasets import load_dataset
from tqdm.auto import tqdm

LABEL_MAP = {
    "reentrancy-no-eth": "reentrancy", "reentrancy-eth": "reentrancy", "safe": "safe",
    "suicidal": "access-control", "arbitrary-send": "access-control",
    "unused-return": "unchecked-calls", "unchecked-lowlevel": "unchecked-calls",
    "msg-value-loop": "double-spending", "controlled-array-length": "access-control",
    "locked-ether": "locked-ether", "tx-origin": "access-control",
    "delegatecall-loop": "double-spending", "constant-function-state": "other",
    "divide-before-multiply": "arithmetic", "array-by-reference": "other",
    "incorrect-equality": "other", "rtlo": "other", "backdoor": "other",
    "weak-prng": "bad-randomness", "controlled-delegatecall": "access-control",
    "constant-function-asm": "other", "unprotected-upgrade": "access-control",
    "uninitialized-storage": "other", "unchecked-send": "locked-ether",
    "uninitialized-local": "other", "unchecked-transfer": "unchecked-calls",
    "mapping-deletion": "other", "uninitialized-state": "other",
}

# batas kelompok dari kuantil panjang file kontrak reentrancy
BATAS = [0, 12389, 18004, 40626, 69888, 240707, 10**12]
TARGET = [50, 75, 125, 125, 75, 50]     # total 500

def kelompok(n):
    for i in range(len(BATAS) - 1):
        if BATAS[i] <= n < BATAS[i + 1]:
            return i
    return len(TARGET) - 1

normal_dir = f'{BASE}/0_normal'
terisi = [0] * len(TARGET)
for f in os.listdir(normal_dir):
    if f.endswith('.sol'):
        terisi[kelompok(os.path.getsize(os.path.join(normal_dir, f)))] += 1
print('sudah ada per kelompok:', terisi)

# jangan ambil alamat yang sudah dipakai di v1
sudah = {f.replace('.sol', '').lower() for f in os.listdir(normal_dir) if f.endswith('.sol')}
lama = '/content/data/raw/dataset_penelitian/0_normal'
if os.path.exists(lama):
    sudah |= {f.replace('.sol', '').lower() for f in os.listdir(lama) if f.endswith('.sol')}
sudah |= {f.replace('.sol', '').lower() for f in os.listdir(f'{BASE}/1_reentrancy')}
print(f'{len(sudah)} alamat dikecualikan')

files = [f"hf://datasets/mwritescode/slither-audited-smart-contracts/data/raw/contracts{i}.parquet"
         for i in range(9)]
ds = load_dataset("parquet", data_files=files, split="train", streaming=True)

ditolak, dilewati_kelompok = 0, 0
pbar = tqdm(total=sum(TARGET) - sum(terisi), desc='kontrak normal')

for item in ds:
    if all(terisi[i] >= TARGET[i] for i in range(len(TARGET))):
        break
    addr_raw = (item.get("contracts") or "").strip()
    sc = item.get("source_code") or ""
    if not addr_raw or not sc:
        continue
    address = f"0x{addr_raw}"
    if address.lower() in sudah:
        continue

    k = kelompok(len(sc.encode('utf-8')))
    if terisi[k] >= TARGET[k]:
        dilewati_kelompok += 1
        continue
    if not kontrak_utuh(sc):
        ditolak += 1
        continue

    try:
        det = (json.loads(item.get("results", "{}")).get("results") or {}).get("detectors", [])
    except Exception:
        det = []
    kelas = {LABEL_MAP[d.get("check")] for d in det if LABEL_MAP.get(d.get("check"))}
    if not det:
        kelas.add("safe")
    if "safe" not in kelas or (kelas - {"safe"}):
        continue

    with open(f'{normal_dir}/{address}.sol', 'w', encoding='utf-8', newline='') as f:
        f.write(sc)
    terisi[k] += 1
    sudah.add(address.lower())
    pbar.update(1)

pbar.close()
print(f'\nterisi per kelompok: {terisi}')
print(f'target             : {TARGET}')
print(f'total normal       : {sum(terisi)}')
print(f'ditolak tidak utuh : {ditolak}')
print(f'dilewati (kuota kelompok penuh): {dilewati_kelompok}')

sudah ada per kelompok: [0, 0, 0, 0, 0, 0]
744 alamat dikecualikan


kontrak normal:   0%|          | 0/500 [00:00<?, ?it/s]


terisi per kelompok: [50, 75, 125, 125, 75, 2]
target             : [50, 75, 125, 125, 75, 50]
total normal       : 452
ditolak tidak utuh : 1511
dilewati (kuota kelompok penuh): 110417


In [ ]:
# === SEL 5: ISI KEKURANGAN KELOMPOK TERBESAR ===
from datasets import load_dataset
from tqdm.auto import tqdm

normal_dir = f'{BASE}/0_normal'
terisi = [0] * len(TARGET)
for f in os.listdir(normal_dir):
    if f.endswith('.sol'):
        terisi[kelompok(os.path.getsize(os.path.join(normal_dir, f)))] += 1
print('terisi sekarang:', terisi)

sudah = {f.replace('.sol', '').lower() for f in os.listdir(normal_dir) if f.endswith('.sol')}
sudah |= {f.replace('.sol', '').lower() for f in os.listdir(f'{BASE}/1_reentrancy')}
lama = '/content/data/raw/dataset_penelitian/0_normal'
if os.path.exists(lama):
    sudah |= {f.replace('.sol', '').lower() for f in os.listdir(lama) if f.endswith('.sol')}

ds = load_dataset("parquet", data_files=files, split="train", streaming=True)
kurang = TARGET[5] - terisi[5]
pbar = tqdm(total=kurang, desc='kontrak besar non-reentrancy')
ditolak2 = 0

for item in ds:
    if terisi[5] >= TARGET[5]:
        break
    sc = item.get("source_code") or ""
    addr_raw = (item.get("contracts") or "").strip()
    if not sc or not addr_raw:
        continue
    if len(sc.encode('utf-8')) < BATAS[5]:
        continue
    address = f"0x{addr_raw}"
    if address.lower() in sudah:
        continue
    if not kontrak_utuh(sc):
        ditolak2 += 1
        continue

    try:
        det = (json.loads(item.get("results", "{}")).get("results") or {}).get("detectors", [])
    except Exception:
        det = []
    kelas = {LABEL_MAP[d.get("check")] for d in det if LABEL_MAP.get(d.get("check"))}
    if "reentrancy" in kelas:          # definisi baru: cukup bukan reentrancy
        continue

    with open(f'{normal_dir}/{address}.sol', 'w', encoding='utf-8', newline='') as f:
        f.write(sc)
    terisi[5] += 1
    sudah.add(address.lower())
    pbar.update(1)

pbar.close()
print(f'\nterisi per kelompok: {terisi}')
print(f'target             : {TARGET}')
print(f'total normal       : {sum(terisi)}')
print(f'ditolak tidak utuh : {ditolak2}')

terisi sekarang: [50, 75, 125, 125, 75, 2]


kontrak besar non-reentrancy:   0%|          | 0/48 [00:00<?, ?it/s]


terisi per kelompok: [50, 75, 125, 125, 75, 50]
target             : [50, 75, 125, 125, 75, 50]
total normal       : 500
ditolak tidak utuh : 3


In [ ]:
# === SEL 6: VALIDASI KOMPILASI KONTRAK NORMAL BARU ===
from slither import Slither
import logging
logging.getLogger('CryticCompile').setLevel(logging.ERROR)

os.chdir('/content')
MANIFEST = f'{D}/data/processed/v2_manifest.csv'

tugas = []
for fn in sorted(os.listdir(f'{BASE}/0_normal')):
    if fn.endswith('.sol'):
        fp = f'{BASE}/0_normal/{fn}'
        pragma = get_pragma(fp)
        tugas.append((resolve_solc(pragma) or 'ZZZ', fp, fn, '0_normal', 0, pragma))
tugas.sort()
print(f'{len(tugas)} kontrak normal akan divalidasi')

sudah_v = set()
if os.path.exists(MANIFEST):
    sudah_v = set(pd.read_csv(MANIFEST).path)
    print(f'{len(sudah_v)} sudah tervalidasi, dilewati')

import csv
baru = not os.path.exists(MANIFEST)
fh = open(MANIFEST, 'a', newline='')
w = csv.writer(fh)
if baru:
    w.writerow(['path','nama_file','folder','label','pragma','solc','status','alasan','n_fungsi','n_node'])

versi_aktif, mulai = None, time.time()
for i, (ver, fp, fn, folder, label, pragma) in enumerate(tugas, 1):
    if fp in sudah_v:
        continue
    status, alasan, nf, nn = 'GAGAL', '', 0, 0
    if ver == 'ZZZ':
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(fp)
            nf = sum(len(c.functions) for c in sl.contracts)
            nn = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
            status = 'OK' if nf > 0 else 'GAGAL'
            if nf == 0:
                alasan = 'tidak_ada_fungsi'
        except Exception as e:
            alasan = str(e).replace('\n',' ')[:200]
    w.writerow([fp, fn, folder, label, pragma, ver if ver!='ZZZ' else '', status, alasan, nf, nn])
    fh.flush()
    if i % 25 == 0:
        print(f'[{i}/{len(tugas)}] {status} {ver} {fn[:26]} ({(time.time()-mulai)/60:.1f} mnt)')

fh.close()
mf = pd.read_csv(MANIFEST)
print(f'\n{len(mf)} baris, OK={sum(mf.status=="OK")}, GAGAL={sum(mf.status=="GAGAL")}')
print(f'waktu {(time.time()-mulai)/60:.1f} menit')

500 kontrak normal akan divalidasi
[25/500] OK 0.4.26 0x02dc5c6317cb81fc1ba5fd7f (0.3 mnt)
[50/500] OK 0.5.17 0x024f4e507d98638a253b59f1 (0.7 mnt)
[75/500] OK 0.5.17 0x33bFB0B95b76f556327ECCb9 (1.1 mnt)
[100/500] OK 0.6.12 0x0245BF267FedC29f28FC91b3 (1.5 mnt)
[125/500] OK 0.6.12 0x03C64A4966845C48D64AA168 (1.8 mnt)
[150/500] OK 0.6.12 0x053ffa1b16124cfc3c70f835 (2.1 mnt)
[175/500] OK 0.6.12 0x0a1237330268ceb2e1a8565b (2.5 mnt)
[200/500] OK 0.6.12 0x265bd427974812123fc1489c (3.3 mnt)
[225/500] GAGAL 0.6.12 0x4a4Bb24dE2dAcF520Fdcc3C2 (3.8 mnt)
[250/500] GAGAL 0.6.12 0x9092147c9414c3b0f6DD487a (4.2 mnt)
[275/500] GAGAL 0.6.6 0x29e1d9A1C97380C8f09b67df (4.6 mnt)
[300/500] GAGAL 0.6.6 0xb27E93A88Efe98A9be54560e (4.9 mnt)
[325/500] OK 0.7.6 0x0540912a921e8e4a212beb8d (5.3 mnt)
[350/500] GAGAL 0.7.6 0x3ADb04E127b9C0a5D3609412 (5.7 mnt)
[375/500] OK 0.8.27 0x021966210719fb00b4552881 (6.1 mnt)
[400/500] OK 0.8.27 0x031a0b8aeeb8bd01bbb1dc3e (6.4 mnt)
[425/500] OK 0.8.27 0x049955995e9f9cc888172a5

In [ ]:
# === SEL 7: PERBAIKI SPDX GANDA, LALU VALIDASI ULANG ===
mf = pd.read_csv(MANIFEST)
kandidat = mf[(mf.status=='GAGAL') & mf.alasan.fillna('').str.contains('SPDX')]
print(f'{len(kandidat)} kontrak dengan masalah SPDX')

def bersihkan_spdx(fp):
    with open(fp, encoding='utf-8', errors='replace') as f:
        baris = f.readlines()
    keluar, sudah_spdx = [], False
    for b in baris:
        if 'SPDX-License-Identifier' in b:
            if sudah_spdx:
                continue
            sudah_spdx = True
        keluar.append(b)
    if not sudah_spdx:
        keluar.insert(0, '// SPDX-License-Identifier: MIT\n')
    with open(fp, 'w', encoding='utf-8', newline='') as f:
        f.writelines(keluar)

hasil = {}
versi_aktif = None
for i, (_, r) in enumerate(kandidat.iterrows(), 1):
    bersihkan_spdx(r.path)
    ver = resolve_solc(get_pragma(r.path))
    status, alasan, nf, nn = 'GAGAL', '', 0, 0
    try:
        if ver != versi_aktif:
            subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
            versi_aktif = ver
        sl = Slither(r.path)
        nf = sum(len(c.functions) for c in sl.contracts)
        nn = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
        status = 'OK' if nf > 0 else 'GAGAL'
    except Exception as e:
        alasan = str(e).replace('\n',' ')[:200]
    hasil[r.path] = (ver, status, alasan, nf, nn)
    print(f'[{i}/{len(kandidat)}] {status} {r.nama_file[:26]} {alasan[:45]}')

pulih = sum(1 for v in hasil.values() if v[1]=='OK')
print(f'\npulih {pulih} dari {len(kandidat)}')

for path,(ver,status,alasan,nf,nn) in hasil.items():
    if status=='OK':
        m = mf.path==path
        mf.loc[m, ['solc','status','alasan','n_fungsi','n_node']] = [ver,status,'',nf,nn]
mf.to_csv(MANIFEST, index=False)
print('OK sekarang:', sum(mf.status=='OK'))

55 kontrak dengan masalah SPDX
[1/55] GAGAL 0x8FdD3fbFEb32b28fb7355551 Invalid compilation:  Invalid solc compilatio
[2/55] OK 0x01ceac209b3d977fcdf13be4 
[3/55] OK 0x022057c3ef9166433750165b 
[4/55] GAGAL 0x02e4fecc3d55d41b936817dc Invalid compilation:  Invalid solc compilatio
[5/55] OK 0x04671bCEad720C0Bd30aC635 
[6/55] OK 0x04969cd041c0cafb6ac462bd 
[7/55] OK 0x054f76beed60ab6dbeb23502 
[8/55] GAGAL 0x08ca538F87668521472e805b Invalid compilation:  Invalid solc compilatio
[9/55] OK 0x0b0b802E873e9f2c18b9dDE3 
[10/55] GAGAL 0x0fDF94751F639cAb6c28d2dC Invalid compilation:  Invalid solc compilatio
[11/55] GAGAL 0x199D3D6B41bced35c45A0156 Invalid compilation:  Invalid solc compilatio
[12/55] OK 0x1a231e75538a931c395785ef 
[13/55] OK 0x1b06b43f3b0bbad73784d858 
[14/55] GAGAL 0x260dE72088Bf46Af7b77FF14 Invalid compilation:  Invalid solc compilatio
[15/55] GAGAL 0x2aD7D86C56b7a09742213e1e Invalid compilation:  Invalid solc compilatio
[16/55] OK 0x2b3a1233a4e7c10696f2ffe0 
[17/55] OK 0x2b9e4

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

os.makedirs(f'{BASE}/0_normal', exist_ok=True)
mf = pd.read_csv(MANIFEST)

# alamat yang dibutuhkan, tanpa awalan 0x, huruf kecil
butuh = {r.nama_file[2:-4].lower(): r.nama_file for _, r in mf.iterrows()}
ada = {f for f in os.listdir(f'{BASE}/0_normal') if f.endswith('.sol')}
butuh = {k: v for k, v in butuh.items() if v not in ada}
print(f'{len(butuh)} kontrak normal perlu diunduh ulang')

files = [f"hf://datasets/mwritescode/slither-audited-smart-contracts/data/raw/contracts{i}.parquet"
         for i in range(9)]
ds = load_dataset("parquet", data_files=files, split="train", streaming=True)

pbar = tqdm(total=len(butuh), desc='unduh ulang')
for item in ds:
    if not butuh:
        break
    a = (item.get("contracts") or "").strip().lower()
    if a in butuh:
        sc = item.get("source_code") or ""
        with open(f'{BASE}/0_normal/{butuh[a]}', 'w', encoding='utf-8', newline='') as f:
            f.write(sc)
        del butuh[a]
        pbar.update(1)
pbar.close()

print(f'\nnormal    : {len([f for f in os.listdir(f"{BASE}/0_normal") if f.endswith(".sol")])}')
print(f'gagal ambil: {len(butuh)}')

500 kontrak normal perlu diunduh ulang


unduh ulang:   0%|          | 0/500 [00:00<?, ?it/s]


normal    : 500
gagal ambil: 0


In [ ]:
import shutil
os.makedirs(f'{BASE}/1_reentrancy', exist_ok=True)
if not os.path.exists('/content/data/raw/dataset_penelitian'):
    !unzip -q -o {D}/data/raw/dataset_final.zip -d /content/data/raw/

src = '/content/data/raw/dataset_penelitian/1_reentrancy'
for f in os.listdir(src):
    if f.endswith('.sol'):
        shutil.copy(os.path.join(src, f), f'{BASE}/1_reentrancy/{f}')
print('reentrancy:', len(os.listdir(f'{BASE}/1_reentrancy')))

reentrancy: 244


In [ ]:
!cd /content/data/raw && zip -qr {D}/data/raw/dataset_v2.zip dataset_v2
!ls -lh {D}/data/raw/

total 26M
-rw------- 1 root root 4.8M Sep 23 07:31 dataset_final.zip
-rw------- 1 root root 7.3M Aug 26 06:37 dataset_penelitian.zip
-rw------- 1 root root  11M Sep 29 06:13 dataset_v2.zip
-rw------- 1 root root 3.2M Sep 23 07:31 karantina.zip


In [ ]:
import os
for f in ['0_normal', '1_reentrancy']:
    p = f'{BASE}/{f}'
    n = len([x for x in os.listdir(p) if x.endswith('.sol')]) if os.path.exists(p) else 0
    print(f'{f}: {n} file')

0_normal: 500 file
1_reentrancy: 244 file


# Perbaikan Resolver Versi Solc

In [ ]:
LABEL_MAP = {
    "reentrancy-no-eth": "reentrancy", "reentrancy-eth": "reentrancy", "safe": "safe",
    "suicidal": "access-control", "arbitrary-send": "access-control",
    "unused-return": "unchecked-calls", "unchecked-lowlevel": "unchecked-calls",
    "msg-value-loop": "double-spending", "controlled-array-length": "access-control",
    "locked-ether": "locked-ether", "tx-origin": "access-control",
    "delegatecall-loop": "double-spending", "constant-function-state": "other",
    "divide-before-multiply": "arithmetic", "array-by-reference": "other",
    "incorrect-equality": "other", "rtlo": "other", "backdoor": "other",
    "weak-prng": "bad-randomness", "controlled-delegatecall": "access-control",
    "constant-function-asm": "other", "unprotected-upgrade": "access-control",
    "uninitialized-storage": "other", "unchecked-send": "locked-ether",
    "uninitialized-local": "other", "unchecked-transfer": "unchecked-calls",
    "mapping-deletion": "other", "uninitialized-state": "other",
}
print('LABEL_MAP siap,', len(LABEL_MAP), 'entri')

LABEL_MAP siap, 28 entri


In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

normal_dir = f'{BASE}/0_normal'
sudah = {f.replace('.sol','').lower() for f in os.listdir(normal_dir) if f.endswith('.sol')}
sudah |= {f.replace('.sol','').lower() for f in os.listdir(f'{BASE}/1_reentrancy')}
print(f'{len(sudah)} alamat dikecualikan')

files = [f"hf://datasets/mwritescode/slither-audited-smart-contracts/data/raw/contracts{i}.parquet"
         for i in range(9)]
ds = load_dataset("parquet", data_files=files, split="train", streaming=True)

BATAS_BESAR = 240707
TARGET_BARU = 400          # ambil banyak, karena tingkat lolos rendah

ditolak, diambil = 0, 0
pbar = tqdm(total=TARGET_BARU, desc='kandidat besar')

for item in ds:
    if diambil >= TARGET_BARU:
        break
    sc = item.get("source_code") or ""
    addr_raw = (item.get("contracts") or "").strip()
    if not sc or not addr_raw:
        continue
    if len(sc.encode('utf-8')) < BATAS_BESAR:
        continue
    address = f"0x{addr_raw}"
    if address.lower() in sudah:
        continue
    if not kontrak_utuh(sc):
        ditolak += 1
        continue

    try:
        det = (json.loads(item.get("results", "{}")).get("results") or {}).get("detectors", [])
    except Exception:
        det = []
    kelas = {LABEL_MAP[d.get("check")] for d in det if LABEL_MAP.get(d.get("check"))}
    if "reentrancy" in kelas:
        continue

    with open(f'{normal_dir}/{address}.sol', 'w', encoding='utf-8', newline='') as f:
        f.write(sc)
    sudah.add(address.lower())
    diambil += 1
    pbar.update(1)

pbar.close()
print(f'\ndiambil: {diambil}, ditolak tidak utuh: {ditolak}')

744 alamat dikecualikan


kandidat besar:   0%|          | 0/400 [00:00<?, ?it/s]


diambil: 45, ditolak tidak utuh: 200


In [ ]:
mf = pd.read_csv(MANIFEST)
salah = mf.alasan.fillna('').str.contains('Slither.*not defined', regex=True)
print(f'{salah.sum()} baris keliru akan dihapus dari manifest')
mf = mf[~salah]
mf.to_csv(MANIFEST, index=False)
print('sisa baris:', len(mf))

45 baris keliru akan dihapus dari manifest
sisa baris: 500


In [ ]:
mf_lama = pd.read_csv(MANIFEST)
sudah_v = set(mf_lama.path)

tugas = []
for fn in sorted(os.listdir(f'{BASE}/0_normal')):
    fp = f'{BASE}/0_normal/{fn}'
    if fp not in sudah_v:
        pragma = get_pragma(fp)
        tugas.append((resolve_solc(pragma) or 'ZZZ', fp, fn, pragma))
tugas.sort()
print(f'{len(tugas)} kandidat baru divalidasi')

baris_baru, versi_aktif, mulai = [], None, time.time()
for i, (ver, fp, fn, pragma) in enumerate(tugas, 1):
    status, alasan, nf, nn = 'GAGAL', '', 0, 0
    if ver == 'ZZZ':
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(fp)
            nf = sum(len(c.functions) for c in sl.contracts)
            nn = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
            status = 'OK' if nf > 0 else 'GAGAL'
        except Exception as e:
            alasan = str(e).replace('\n',' ')[:150]
    baris_baru.append([fp, fn, '0_normal', 0, pragma, ver if ver!='ZZZ' else '', status, alasan, nf, nn])
    print(f'[{i}/{len(tugas)}] {status} {ver} {fn[:26]} {alasan[:40]}')

df_baru = pd.DataFrame(baris_baru, columns=mf_lama.columns)
mf_gab = pd.concat([mf_lama, df_baru], ignore_index=True)
mf_gab.to_csv(MANIFEST, index=False)

print(f'\ntotal baru lolos: {sum(df_baru.status=="OK")} dari {len(df_baru)}')
print(f'total OK sekarang: {sum(mf_gab.status=="OK")}')

45 kandidat baru divalidasi
[1/45] OK 0.4.26 0x3f21b20a9ee87c5e6945117e 
[2/45] OK 0.4.26 0x5f2539a5dc504493ba9ae120 
[3/45] OK 0.4.26 0xbfa82fbe0e66d8e2b7dcc163 
[4/45] OK 0.4.26 0xcc6a0dd80ff3fa8c1b370117 
[5/45] GAGAL 0.6.12 0x82991034cCB671F0F6CCFa73 Invalid compilation:  Invalid solc compi
[6/45] GAGAL 0.6.12 0x83719f6408938bde08944819 Invalid compilation:  Invalid solc compi
[7/45] GAGAL 0.6.12 0x9CA23dc1C06cA75C2956381a Invalid compilation:  Invalid solc compi
[8/45] GAGAL 0.6.12 0xB414a17dd9DAb4F6E9fC7BBc Invalid compilation:  Invalid solc compi
[9/45] GAGAL 0.6.12 0xC7cf1DFA6779E2e2E4b92701 Invalid compilation:  Invalid solc compi
[10/45] GAGAL 0.6.12 0xC8446F4244Fa9dDc90B3bB57 Invalid compilation:  Invalid solc compi
[11/45] GAGAL 0.6.12 0xD0eB57ff3eA4Def2b74dc296 Invalid compilation:  Invalid solc compi
[12/45] GAGAL 0.6.12 0xD6F7171a0f873262267CF99e Invalid compilation:  Invalid solc compi
[13/45] GAGAL 0.6.12 0xE0cA50b81442C2Fe2b20D0A0 Invalid compilation:  Invalid solc co

In [ ]:
def resolve_solc_v2(pragma_str):
    if not pragma_str or str(pragma_str) == 'TIDAK_DITEMUKAN':
        return None
    s = str(pragma_str).replace('^', '~=').strip()
    # satukan operator dan angka yang terpisah spasi, contoh ">= 0.4.22" -> ">=0.4.22"
    s = re.sub(r'([<>=~!]+)\s+', r'\1', s)
    try:
        spec = SpecifierSet(','.join(p.strip() for p in s.split() if p.strip()))
        cocok = [v for v in terpasang if v in spec]
        if cocok:
            return str(max(cocok))
    except Exception:
        pass
    m = re.search(r'(\d+\.\d+\.\d+)', str(pragma_str))
    return m.group(1) if m else None

In [ ]:
mf = pd.read_csv(MANIFEST)
sisa_semua = mf[(mf.status=='GAGAL') & (mf.folder=='0_normal')]
print(f'{len(sisa_semua)} kontrak akan dicoba ulang')

hasil, versi_aktif = {}, None
for i, (_, r) in enumerate(sisa_semua.iterrows(), 1):
    bersihkan_spdx(r.path)
    ver = resolve_solc_v2(get_pragma(r.path))
    status, alasan, nf, nn = 'GAGAL', '', 0, 0
    try:
        if ver != versi_aktif:
            subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
            versi_aktif = ver
        sl = Slither(r.path)
        nf = sum(len(c.functions) for c in sl.contracts)
        nn = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
        status = 'OK' if nf > 0 else 'GAGAL'
    except Exception as e:
        alasan = str(e).replace('\n',' ')[:150]
    hasil[r.path] = (ver, status, alasan, nf, nn)
    if i % 30 == 0:
        pulih = sum(1 for v in hasil.values() if v[1]=='OK')
        print(f'[{i}/{len(sisa_semua)}] pulih sejauh ini: {pulih}')

pulih = sum(1 for v in hasil.values() if v[1]=='OK')
print(f'\npulih total: {pulih} dari {len(sisa_semua)}')

for path,(ver,status,alasan,nf,nn) in hasil.items():
    if status=='OK':
        m = mf.path==path
        mf.loc[m, ['solc','status','alasan','n_fungsi','n_node']] = [ver,status,'',nf,nn]
mf.to_csv(MANIFEST, index=False)
print('OK sekarang:', sum(mf.status=='OK'))

167 kontrak akan dicoba ulang
[30/167] pulih sejauh ini: 1
[60/167] pulih sejauh ini: 1
[90/167] pulih sejauh ini: 1
[120/167] pulih sejauh ini: 1
[150/167] pulih sejauh ini: 1

pulih total: 1 dari 167
OK sekarang: 379


In [ ]:
def resolve_solc_multi(fp):
    with open(fp, encoding='utf-8', errors='replace') as f:
        isi = f.read()
    semua = re.findall(r'pragma\s+solidity\s*([^;]+);', isi)
    if not semua:
        return None
    kandidat = [resolve_solc_v2(p.strip()) for p in semua]
    kandidat = [Version(v) for v in kandidat if v]
    return str(max(kandidat)) if kandidat else None

# uji pada kontrak yang sama
print('versi terpilih (multi-pragma):', resolve_solc_multi(r.path))

versi terpilih (multi-pragma): 0.6.12


In [ ]:
mf = pd.read_csv(MANIFEST)
sisa_semua = mf[(mf.status=='GAGAL') & (mf.folder=='0_normal')]
print(f'{len(sisa_semua)} kontrak akan dicoba ulang')

hasil, versi_aktif = {}, None
for i, (_, r) in enumerate(sisa_semua.iterrows(), 1):
    ver = resolve_solc_multi(r.path)
    status, alasan, nf, nn = 'GAGAL', '', 0, 0
    if not ver:
        alasan = 'pragma_tidak_ditemukan'
    else:
        try:
            if ver != versi_aktif:
                subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
                versi_aktif = ver
            sl = Slither(r.path)
            nf = sum(len(c.functions) for c in sl.contracts)
            nn = sum(len(f.nodes) for c in sl.contracts for f in c.functions)
            status = 'OK' if nf > 0 else 'GAGAL'
        except Exception as e:
            alasan = str(e).replace('\n',' ')[:150]
    hasil[r.path] = (ver, status, alasan, nf, nn)
    if i % 30 == 0:
        pulih = sum(1 for v in hasil.values() if v[1]=='OK')
        print(f'[{i}/{len(sisa_semua)}] pulih sejauh ini: {pulih}')

pulih = sum(1 for v in hasil.values() if v[1]=='OK')
print(f'\npulih total: {pulih} dari {len(sisa_semua)}')

for path,(ver,status,alasan,nf,nn) in hasil.items():
    if status=='OK':
        m = mf.path==path
        mf.loc[m, ['solc','status','alasan','n_fungsi','n_node']] = [ver,status,'',nf,nn]
    else:
        m = mf.path==path
        mf.loc[m, 'alasan'] = alasan
mf.to_csv(MANIFEST, index=False)
print('OK sekarang:', sum(mf.status=='OK'))

166 kontrak akan dicoba ulang
[30/166] pulih sejauh ini: 1
[60/166] pulih sejauh ini: 1
[90/166] pulih sejauh ini: 1
[120/166] pulih sejauh ini: 1
[150/166] pulih sejauh ini: 1

pulih total: 1 dari 166
OK sekarang: 380


# Karantina kontrak gagal dan Verifikasi Dataset Final

In [ ]:
import shutil
KARANTINA_V2 = '/content/data/raw/karantina_v2'
os.makedirs(KARANTINA_V2, exist_ok=True)

mf = pd.read_csv(MANIFEST)
gagal = mf[(mf.status=='GAGAL') & (mf.folder=='0_normal')]
for _, r in gagal.iterrows():
    if os.path.exists(r.path):
        shutil.move(r.path, f'{KARANTINA_V2}/{r.nama_file}')

print(f'{len(gagal)} kontrak dipindah ke karantina')
print(f'tersisa 0_normal: {len([f for f in os.listdir(f"{BASE}/0_normal") if f.endswith(".sol")])}')

165 kontrak dipindah ke karantina
tersisa 0_normal: 380


In [ ]:
mf = pd.read_csv(MANIFEST)
ok = mf[mf.status=='OK'].copy()
ren = pd.read_csv(f'{D}/data/processed/ukuran_reentrancy.csv')

gab = pd.concat([
    pd.DataFrame({'n_node': ok.n_node, 'label': 0}),
    pd.DataFrame({'n_node': ren.n_node, 'label': 1}),
])
print(f'normal: {len(ok)}   reentrancy: {len(ren)}   rasio: {len(ok)/len(ren):.2f}\n')
for lo, hi in [(0,50),(50,150),(150,500),(500,1500),(1500,10**9)]:
    m = (gab.n_node >= lo) & (gab.n_node < hi)
    if m.sum():
        print(f'  node {lo:>5}-{hi:<8} n={m.sum():3}  rentan={gab.label[m].mean():.2f}')

normal: 380   reentrancy: 244   rasio: 1.56

  node     0-50       n= 27  rentan=0.15
  node    50-150      n= 68  rentan=0.21
  node   150-500      n=330  rentan=0.35
  node   500-1500     n=153  rentan=0.42
  node  1500-1000000000 n= 46  rentan=0.98


# Ekstraksi CFG Kontrak Normal

In [ ]:
import time
from slither import Slither

TIPE_NODE = ['ENTRYPOINT','EXPRESSION','RETURN','VARIABLE','IF','ENDIF',
             'OTHER_ENTRYPOINT','STARTLOOP','ENDLOOP','IFLOOP','ASSEMBLY',
             'ENDASSEMBLY','BREAK','CONTINUE','CATCH','TRY','THROW']

def ekstrak_cfg(fp):
    sl = Slither(fp)
    peta, nodes, fungsi_entry = {}, [], {}
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                gid = len(nodes)
                peta[(c.name, fn.full_name, n.node_id)] = gid
                tipe = str(n.type).replace('NodeType.', '')
                nodes.append({
                    'id': gid, 'tipe': tipe,
                    'n_call_low': len(n.low_level_calls),
                    'n_call_high': len(n.high_level_calls),
                    'n_internal': len(n.internal_calls),
                    'n_sstore': len(n.state_variables_written),
                    'n_sload': len(n.state_variables_read),
                    'n_irs': len(n.irs),
                })
                if n.node_id == 0:
                    fungsi_entry[(c.name, fn.full_name)] = gid

    edges = []
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                a = peta[(c.name, fn.full_name, n.node_id)]
                for m in n.sons:
                    k = (c.name, fn.full_name, m.node_id)
                    if k in peta:
                        edges.append([a, peta[k], 0])
                for ic in n.internal_calls:
                    target = getattr(ic, 'function', ic)
                    nama = getattr(target, 'full_name', None)
                    if nama and (c.name, nama) in fungsi_entry:
                        edges.append([a, fungsi_entry[(c.name, nama)], 1])

    return {'nodes': nodes, 'edges': edges, 'n_node': len(nodes), 'n_edge': len(edges)}

OUT_CFG = f'{D}/data/processed/v2_cfg'
os.makedirs(OUT_CFG, exist_ok=True)

os.chdir('/content')
mf = pd.read_csv(MANIFEST)
df_ok = mf[mf.status=='OK'].copy().sort_values('solc')
sudah = {f[:-5] for f in os.listdir(OUT_CFG) if f.endswith('.json')}
print(f'{len(df_ok)} kontrak normal akan diekstraksi, {len(sudah)} sudah ada')

versi_aktif, berhasil, gagal2, mulai = None, 0, [], time.time()
for i, (_, r) in enumerate(df_ok.iterrows(), 1):
    kunci = f"0_normal__{r.nama_file[:-4]}"
    if kunci in sudah:
        continue
    try:
        if r.solc != versi_aktif:
            subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
            versi_aktif = r.solc
        g = ekstrak_cfg(r.path)
        g.update({'nama_file': r.nama_file, 'folder': '0_normal', 'label': 0, 'split': 'train'})
        with open(f'{OUT_CFG}/{kunci}.json','w') as f:
            json.dump(g, f)
        berhasil += 1
        if i % 50 == 0:
            print(f'[{i}/{len(df_ok)}] ({(time.time()-mulai)/60:.1f} mnt)')
    except Exception as e:
        gagal2.append((r.nama_file, str(e)[:100]))

print(f'\nEkstraksi normal selesai: {berhasil} berhasil, {len(gagal2)} gagal, {(time.time()-mulai)/60:.1f} menit')

380 kontrak normal akan diekstraksi, 0 sudah ada
[50/380] (0.7 mnt)
[100/380] (1.9 mnt)
[200/380] (3.8 mnt)
[250/380] (4.6 mnt)
[300/380] (5.4 mnt)
[350/380] (6.3 mnt)

Ekstraksi normal selesai: 367 berhasil, 13 gagal, 6.7 menit


In [ ]:
mf = pd.read_csv(MANIFEST)
df_ok = mf[mf.status=='OK'].copy()

diekstrak = {f[:-5].replace('0_normal__','') for f in os.listdir(OUT_CFG) if f.startswith('0_normal__')}
belum = df_ok[~df_ok.nama_file.str.replace('.sol','',regex=False).isin(diekstrak)]
print(f'{len(belum)} kontrak normal gagal diekstraksi CFG (meski lolos validasi kompilasi)\n')

for _, r in belum.head(13).iterrows():
    try:
        subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
        ekstrak_cfg(r.path)
        print(r.nama_file[:30], '-> ternyata berhasil sekarang')
    except Exception as e:
        print(r.nama_file[:30], '->', str(e)[:100])

13 kontrak normal gagal diekstraksi CFG (meski lolos validasi kompilasi)

0x01ceac209b3d977fcdf13be403ab -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identifiers found in sou
0x022057c3ef9166433750165bfda2 -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identifiers found in sou
0x04671bCEad720C0Bd30aC6350145 -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identifiers found in sou
0x04969cd041c0cafb6ac462bd65b5 -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identifiers found in sou
0x054f76beed60ab6dbeb23502178c -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identifiers found in sou
0x0b0b802E873e9f2c18b9dDE3AE7f -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identifiers found in sou
0x1a231e75538a931c395785ef5d1a -> Invalid compilation: 
Invalid solc compilation Error: Multiple SPDX license identi

In [ ]:
for _, r in belum.iterrows():
    bersihkan_spdx(r.path)

# coba ekstraksi ulang
gagal_lagi = []
for _, r in belum.iterrows():
    try:
        subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
        ekstrak_cfg(r.path)
        print(r.nama_file[:30], '-> OK setelah dibersihkan')
    except Exception as e:
        gagal_lagi.append(r.nama_file)
        print(r.nama_file[:30], '-> masih gagal')

print(f'\nmasih gagal: {len(gagal_lagi)} dari {len(belum)}')

0x01ceac209b3d977fcdf13be403ab -> OK setelah dibersihkan
0x022057c3ef9166433750165bfda2 -> OK setelah dibersihkan
0x04671bCEad720C0Bd30aC6350145 -> OK setelah dibersihkan
0x04969cd041c0cafb6ac462bd65b5 -> OK setelah dibersihkan
0x054f76beed60ab6dbeb23502178c -> OK setelah dibersihkan
0x0b0b802E873e9f2c18b9dDE3AE7f -> OK setelah dibersihkan
0x1a231e75538a931c395785ef5d1a -> OK setelah dibersihkan
0x1b06b43f3b0bbad73784d858c782 -> OK setelah dibersihkan
0x2b3a1233a4e7c10696f2ffe05cce -> OK setelah dibersihkan
0x2b9e49ebd85ec3b2cded8818415f -> OK setelah dibersihkan
0x2f9f75289466e9005f44c4fa25dd -> OK setelah dibersihkan
0x2fbc5d9a40de4da5d930e55d7391 -> OK setelah dibersihkan
0x43044f861ec040db59a7e324c405 -> OK setelah dibersihkan

masih gagal: 0 dari 13


# Ekstraksi CFG Kontrak Reentrancy

In [ ]:
# === SETUP LANJUT EKSTRAKSI CFG ===
from google.colab import drive
drive.mount('/content/drive')
!pip install -q slither-analyzer solc-select

import os, re, json, time, subprocess
import pandas as pd
from packaging.specifiers import SpecifierSet
from packaging.version import Version
from slither import Slither
from solc_select import solc_select
import logging
logging.getLogger('CryticCompile').setLevel(logging.ERROR)

D = '/content/drive/MyDrive/PROJECT'
BASE = '/content/data/raw/dataset_v2'
OUT_CFG = f'{D}/data/processed/v2_cfg'
MANIFEST = f'{D}/data/processed/v2_manifest.csv'

# kembalikan binary solc
if not os.path.exists(os.path.expanduser('~/.solc-select/artifacts')):
    os.makedirs(os.path.expanduser('~/.solc-select'), exist_ok=True)
    !cp -r {D}/solc_binaries/artifacts ~/.solc-select/
    !chmod -R +x ~/.solc-select/artifacts/

# kembalikan file .sol dari zip
if not os.path.exists(f'{BASE}/1_reentrancy'):
    os.makedirs('/content/data/raw', exist_ok=True)
    !unzip -q -o {D}/data/raw/dataset_v2.zip -d /content/data/raw/

terpasang = sorted([Version(v) for v in solc_select.installed_versions()])

def get_pragma(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        c = f.read()
    m = re.search(r'pragma\s+solidity\s*([^;]+);', c)
    return m.group(1).strip() if m else 'TIDAK_DITEMUKAN'

def resolve_solc_v2(pragma_str):
    if not pragma_str or str(pragma_str) == 'TIDAK_DITEMUKAN':
        return None
    s = str(pragma_str).replace('^', '~=').strip()
    s = re.sub(r'([<>=~!]+)\s+', r'\1', s)
    try:
        spec = SpecifierSet(','.join(p.strip() for p in s.split() if p.strip()))
        cocok = [v for v in terpasang if v in spec]
        if cocok:
            return str(max(cocok))
    except Exception:
        pass
    m = re.search(r'(\d+\.\d+\.\d+)', str(pragma_str))
    return m.group(1) if m else None

def resolve_solc_multi(fp):
    with open(fp, encoding='utf-8', errors='replace') as f:
        isi = f.read()
    semua = re.findall(r'pragma\s+solidity\s*([^;]+);', isi)
    if not semua:
        return None
    kandidat = [resolve_solc_v2(p.strip()) for p in semua]
    kandidat = [Version(v) for v in kandidat if v]
    return str(max(kandidat)) if kandidat else None

def ekstrak_cfg(fp):
    sl = Slither(fp)
    peta, nodes, fungsi_entry = {}, [], {}
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                gid = len(nodes)
                peta[(c.name, fn.full_name, n.node_id)] = gid
                nodes.append({
                    'id': gid, 'tipe': str(n.type).replace('NodeType.', ''),
                    'n_call_low': len(n.low_level_calls),
                    'n_call_high': len(n.high_level_calls),
                    'n_internal': len(n.internal_calls),
                    'n_sstore': len(n.state_variables_written),
                    'n_sload': len(n.state_variables_read),
                    'n_irs': len(n.irs),
                })
                if n.node_id == 0:
                    fungsi_entry[(c.name, fn.full_name)] = gid
    edges = []
    for c in sl.contracts:
        for fn in c.functions:
            for n in fn.nodes:
                a = peta[(c.name, fn.full_name, n.node_id)]
                for m in n.sons:
                    k = (c.name, fn.full_name, m.node_id)
                    if k in peta:
                        edges.append([a, peta[k], 0])
                for ic in n.internal_calls:
                    target = getattr(ic, 'function', ic)
                    nama = getattr(target, 'full_name', None)
                    if nama and (c.name, nama) in fungsi_entry:
                        edges.append([a, fungsi_entry[(c.name, nama)], 1])
    return {'nodes': nodes, 'edges': edges, 'n_node': len(nodes), 'n_edge': len(edges)}

n_norm = len([f for f in os.listdir(OUT_CFG) if f.startswith('0_normal__')])
n_ren = len([f for f in os.listdir(OUT_CFG) if f.startswith('1_reentrancy__')])
print(f'siap. CFG tersimpan: {n_norm} normal, {n_ren} reentrancy')
print(f'solc terpasang: {len(terpasang)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
siap. CFG tersimpan: 367 normal, 61 reentrancy
solc terpasang: 61


In [ ]:
print('BASE ada        :', os.path.exists(BASE))
print('folder reentrancy:', os.path.exists(f'{BASE}/1_reentrancy'))
if os.path.exists(f'{BASE}/1_reentrancy'):
    print('jumlah .sol     :', len([f for f in os.listdir(f'{BASE}/1_reentrancy') if f.endswith('.sol')]))
print('zip di Drive ada:', os.path.exists(f'{D}/data/raw/dataset_v2.zip'))

BASE ada        : True
folder reentrancy: True
jumlah .sol     : 0
zip di Drive ada: True


In [ ]:
os.makedirs('/content/data/raw', exist_ok=True)
!unzip -q -o {D}/data/raw/dataset_v2.zip -d /content/data/raw/
!ls /content/data/raw/
print('reentrancy:', len([f for f in os.listdir(f'{BASE}/1_reentrancy') if f.endswith('.sol')]))
print('normal    :', len([f for f in os.listdir(f'{BASE}/0_normal') if f.endswith('.sol')]))

dataset_v2
reentrancy: 244
normal    : 500


In [ ]:
df_ren = pd.DataFrame([{
    'path': f'{BASE}/1_reentrancy/{f}',
    'nama_file': f,
    'solc': resolve_solc_multi(f'{BASE}/1_reentrancy/{f}')
} for f in os.listdir(f'{BASE}/1_reentrancy') if f.endswith('.sol')])
df_ren = df_ren.sort_values('solc')
print(f'{len(df_ren)} kontrak reentrancy')

244 kontrak reentrancy


In [ ]:
n_cfg = len([f for f in os.listdir(OUT_CFG) if f.endswith('.json')])
n_normal = len([f for f in os.listdir(OUT_CFG) if f.startswith('0_normal__')])
n_ren = len([f for f in os.listdir(OUT_CFG) if f.startswith('1_reentrancy__')])
print(f'total CFG tersimpan : {n_cfg}')
print(f'  normal            : {n_normal}')
print(f'  reentrancy        : {n_ren}')

total CFG tersimpan : 428
  normal            : 367
  reentrancy        : 61


In [ ]:
sudah = {f[:-5] for f in os.listdir(OUT_CFG) if f.endswith('.json')}
versi_aktif, berhasil2, gagal3, mulai = None, 0, [], time.time()
for i, (_, r) in enumerate(df_ren.iterrows(), 1):
    kunci = f"1_reentrancy__{r.nama_file[:-4]}"
    if kunci in sudah:
        continue
    try:
        if r.solc != versi_aktif:
            subprocess.run(['solc-select','use',str(r.solc)], capture_output=True, check=True)
            versi_aktif = r.solc
        g = ekstrak_cfg(r.path)
        g.update({'nama_file': r.nama_file, 'folder': '1_reentrancy', 'label': 1, 'split': 'train'})
        with open(f'{OUT_CFG}/{kunci}.json','w') as f:
            json.dump(g, f)
        berhasil2 += 1
        if i % 10 == 0:      # dilaporkan tiap 10, bukan 50
            print(f'[{i}/{len(df_ren)}] {r.nama_file[:24]} node={g["n_node"]} ({(time.time()-mulai)/60:.1f} mnt)')
    except Exception as e:
        gagal3.append((r.nama_file, str(e)[:100]))
        print(f'[{i}/{len(df_ren)}] GAGAL {r.nama_file[:24]}')

print(f'\nreentrancy selesai: {berhasil2} berhasil, {len(gagal3)} gagal, {(time.time()-mulai)/60:.1f} menit')

[20/244] GAGAL 0x0046f822ef4e6d51a40dd8
[70/244] 0xa06cd5c489c59b463f8fe0 node=5561 (1.0 mnt)
[80/244] 0x00c83aeCC790e8a4453e5d node=2788 (2.1 mnt)
[90/244] 0xb9cfc55a243CE278d318Bd node=955 (3.2 mnt)
[100/244] 0x008692a1683b7751af28ea node=378 (3.4 mnt)
[110/244] 0xbbdE1047EE332A3798AD90 node=1333 (3.7 mnt)
[120/244] 0xf4128B00AFdA933428056d node=437 (3.9 mnt)
[130/244] 0xa17a8883dA1aBd57c690DF node=305 (4.1 mnt)
[140/244] 0x92c46012e3a7fb21264271 node=199 (4.3 mnt)
[150/244] 0xe18d9062e61e2dcb459601 node=3457 (4.6 mnt)
[160/244] 0xFE721DD8Ac8e47A4228A61 node=606 (4.9 mnt)
[170/244] 0x00da328ea462f75325770a node=547 (5.0 mnt)
[180/244] 0x006aa3eb5e666d8e006aa6 node=237 (5.2 mnt)
[190/244] 0x00d97facb316f12b64c7ac node=213 (5.4 mnt)
[200/244] 0x0105564c5a9054b147d99b node=495 (5.6 mnt)
[201/244] GAGAL 0x0011eaaaa14eb99b1dc0c3
[210/244] 0x00a103267A22971375C3C3 node=412 (5.7 mnt)
[220/244] 0x006277d659707f787fe90e node=479 (5.9 mnt)
[224/244] GAGAL 0x00ef721095c355edf851be
[230/244] 0x0

In [ ]:
for nama, err in gagal3:
    print(nama[:30], '->', err)

mf1 = pd.read_csv(f'{D}/data/processed/manifest_final.csv')
mf1['nama'] = mf1.nama_file
for nama, _ in gagal3:
    baris = mf1[mf1.nama == nama]
    versi_v1 = baris.solc.iloc[0] if len(baris) else None
    versi_sekarang = resolve_solc_multi(f'{BASE}/1_reentrancy/{nama}')
    print(f'{nama[:24]}  v1={versi_v1}  sekarang={versi_sekarang}')

0x0046f822ef4e6d51a40dd87d34ab -> Invalid compilation: 
Invalid solc compilation /content/data/raw/dataset_v2/1_reentrancy/0x0046f822e
0x0011eaaaa14eb99b1dc0c3f931d9 -> Invalid compilation: 
Invalid solc compilation Error: Source file requires different compiler versio
0x00ef721095c355edf851be7ae0a1 -> Invalid compilation: 
Invalid solc compilation Error: Source file requires different compiler versio
0x0046f822ef4e6d51a40dd8  v1=0.4.11  sekarang=0.4.26
0x0011eaaaa14eb99b1dc0c3  v1=0.6.12  sekarang=0.8.27
0x00ef721095c355edf851be  v1=0.6.12  sekarang=0.8.27


In [ ]:
for nama, _ in gagal3:
    baris = mf1[mf1.nama == nama]
    if not len(baris):
        continue
    ver = str(baris.solc.iloc[0])
    fp = f'{BASE}/1_reentrancy/{nama}'
    try:
        subprocess.run(['solc-select','use',ver], capture_output=True, check=True)
        g = ekstrak_cfg(fp)
        g.update({'nama_file': nama, 'folder': '1_reentrancy', 'label': 1, 'split': 'train'})
        with open(f'{OUT_CFG}/1_reentrancy__{nama[:-4]}.json','w') as f:
            json.dump(g, f)
        print(nama[:24], '-> OK dengan versi', ver)
    except Exception as e:
        print(nama[:24], '-> masih gagal:', str(e)[:100])

n_ren = len([f for f in os.listdir(OUT_CFG) if f.startswith('1_reentrancy__')])
print('total reentrancy tersimpan:', n_ren)

0x0046f822ef4e6d51a40dd8 -> OK dengan versi 0.4.11
0x0011eaaaa14eb99b1dc0c3 -> OK dengan versi 0.6.12
0x00ef721095c355edf851be -> OK dengan versi 0.6.12
total reentrancy tersimpan: 244


In [ ]:
import shutil
sumber = f'{D}/data/processed/cfg'
n = 0
for f in os.listdir(sumber):
    if f.startswith('3_expert_benchmark__'):
        shutil.copy(f'{sumber}/{f}', f'{OUT_CFG}/{f}')
        n += 1
print(f'{n} file benchmark disalin ke v2_cfg')

n_total = len([f for f in os.listdir(OUT_CFG) if f.endswith('.json')])
n_norm = len([f for f in os.listdir(OUT_CFG) if f.startswith('0_normal__')])
n_ren = len([f for f in os.listdir(OUT_CFG) if f.startswith('1_reentrancy__')])
n_bm = len([f for f in os.listdir(OUT_CFG) if f.startswith('3_expert_benchmark__')])
print(f'\ntotal CFG v2: {n_total}')
print(f'  normal     : {n_norm}')
print(f'  reentrancy : {n_ren}')
print(f'  benchmark  : {n_bm}')

16 file benchmark disalin ke v2_cfg

total CFG v2: 627
  normal     : 367
  reentrancy : 244
  benchmark  : 16


# Pruning dan Konversi ke PyTorch Geometric

In [ ]:
import math
from collections import Counter

def hitung_idf(cfg_dir):
    df_tipe = Counter()
    n_dok = 0
    for f in os.listdir(cfg_dir):
        if not f.endswith('.json'):
            continue
        n_dok += 1
        with open(f'{cfg_dir}/{f}') as fh:
            tipe_ada = {n['tipe'] for n in json.load(fh)['nodes']}
        for t in tipe_ada:
            df_tipe[t] += 1
    return {t: math.log(n_dok / d) for t, d in df_tipe.items()}, n_dok

def wajib_simpan(n):
    if n['n_call_low'] + n['n_call_high'] > 0:
        return True
    if n['n_sstore'] > 0:
        return True
    if n['tipe'] in ('IF', 'IFLOOP'):
        return True
    if n['tipe'] in ('ENTRYPOINT', 'OTHER_ENTRYPOINT'):
        return True
    if n['n_internal'] > 0:
        return True
    return False

def prune_graf(g, idf, ambang):
    tf = Counter(n['tipe'] for n in g['nodes'])
    total = max(len(g['nodes']), 1)
    simpan = set()
    for n in g['nodes']:
        if wajib_simpan(n):
            simpan.add(n['id'])
            continue
        skor = (tf[n['tipe']] / total) * idf.get(n['tipe'], 0.0)
        if skor >= ambang:
            simpan.add(n['id'])

    keluar = {}
    for a, b, t in g['edges']:
        keluar.setdefault(a, []).append((b, t))

    def telusuri(mulai, tipe_sisi, terlihat):
        hasil = []
        for b, t in keluar.get(mulai, []):
            if b in terlihat:
                continue
            terlihat.add(b)
            if b in simpan:
                hasil.append((b, tipe_sisi))
            else:
                hasil.extend(telusuri(b, tipe_sisi, terlihat))
        return hasil

    edges_baru = set()
    for a, b, t in g['edges']:
        if a not in simpan:
            continue
        if b in simpan:
            edges_baru.add((a, b, t))
        else:
            for c, tt in telusuri(b, t, {a, b}):
                edges_baru.add((a, c, tt))

    urut = sorted(simpan)
    peta = {lama: baru for baru, lama in enumerate(urut)}
    asal = {n['id']: n for n in g['nodes']}
    nodes_baru = []
    for lama in urut:
        n = dict(asal[lama])
        n['id'] = peta[lama]
        nodes_baru.append(n)

    hasil = dict(g)
    hasil['nodes'] = nodes_baru
    hasil['edges'] = [[peta[a], peta[b], t] for a, b, t in sorted(edges_baru)]
    hasil['n_node'] = len(nodes_baru)
    hasil['n_edge'] = len(hasil['edges'])
    return hasil

print('fungsi pruning siap')

fungsi pruning siap


In [ ]:
import sys
sys.setrecursionlimit(20000)

AMBANG = 0.02
OUT_PRUNE = f'{D}/data/processed/v2_cfg_prune_{AMBANG}'
os.makedirs(OUT_PRUNE, exist_ok=True)

idf, n_dok = hitung_idf(OUT_CFG)
print(f'IDF dari {n_dok} graf')

mulai = time.time()
ringkas = []
for f in sorted(os.listdir(OUT_CFG)):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_CFG}/{f}') as fh:
        g = json.load(fh)
    p = prune_graf(g, idf, AMBANG)
    with open(f'{OUT_PRUNE}/{f}', 'w') as fh:
        json.dump(p, fh)
    ringkas.append({'nama': g['nama_file'], 'asli': g['n_node'], 'prune': p['n_node']})

rk = pd.DataFrame(ringkas)
print(f'\n{len(rk)} graf dipangkas dalam {time.time()-mulai:.1f} detik')
print(f'node total: {rk.asli.sum()} -> {rk.prune.sum()} ({rk.prune.sum()/rk.asli.sum()*100:.1f}%)')

IDF dari 627 graf

627 graf dipangkas dalam 13.2 detik
node total: 399088 -> 274174 (68.7%)


In [ ]:
def baca_p(nama):
    with open(f'{OUT_PRUNE}/{nama}.json') as f:
        return json.load(f)

pasangan = [
    ('simple_dao', '3_expert_benchmark__rentan__simple_dao', '3_expert_benchmark__aman__simple_dao_fixed'),
    ('reentrance', '3_expert_benchmark__rentan__reentrance', '3_expert_benchmark__aman__reentrance_fixed'),
    ('etherstore', '3_expert_benchmark__rentan__etherstore', '3_expert_benchmark__aman__etherstore_fixed'),
    ('etherbank',  '3_expert_benchmark__rentan__etherbank',  '3_expert_benchmark__aman__etherbank_fixed'),
]

def sidik(g):
    return [(n['tipe'], int(n['n_call_low']+n['n_call_high']>0), int(n['n_sstore']>0))
            for n in sorted(g['nodes'], key=lambda x: x['id'])]

for nama, kr, ka in pasangan:
    r, a = baca_p(kr), baca_p(ka)
    sama_fitur = sidik(r) == sidik(a)
    print(f'{nama:14} rentan {r["n_node"]}n  aman {a["n_node"]}n  '
          f'{"IDENTIK - BAHAYA" if sama_fitur else "berbeda - OK"}')

simple_dao     rentan 7n  aman 7n  berbeda - OK
reentrance     rentan 10n  aman 10n  berbeda - OK
etherstore     rentan 10n  aman 10n  berbeda - OK
etherbank      rentan 7n  aman 7n  berbeda - OK


In [ ]:
TIPE = ['EXPRESSION','ENTRYPOINT','RETURN','VARIABLE','IF','ENDIF',
        'OTHER_ENTRYPOINT','STARTLOOP','ENDLOOP','IFLOOP','ASSEMBLY',
        'ENDASSEMBLY','BREAK','CONTINUE','CATCH','TRY','THROW']
TIPE_IDX = {t: i for i, t in enumerate(TIPE)}
DIM = len(TIPE) + 5

def vektor_node(n):
    v = [0.0] * DIM
    idx = TIPE_IDX.get(n['tipe'])
    if idx is not None:
        v[idx] = 1.0
    o = len(TIPE)
    v[o+0] = math.log1p(n['n_call_low'])
    v[o+1] = math.log1p(n['n_call_high'])
    v[o+2] = math.log1p(n['n_sstore'])
    v[o+3] = math.log1p(n['n_sload'])
    v[o+4] = math.log1p(n['n_irs'])
    return v

import torch
from torch_geometric.data import Data

def ke_pyg(g):
    nodes = sorted(g['nodes'], key=lambda x: x['id'])
    x = torch.tensor([vektor_node(n) for n in nodes], dtype=torch.float)
    if g['edges']:
        ei = torch.tensor([[e[0] for e in g['edges']], [e[1] for e in g['edges']]], dtype=torch.long)
        et = torch.tensor([e[2] for e in g['edges']], dtype=torch.long)
    else:
        ei = torch.empty((2,0), dtype=torch.long)
        et = torch.empty((0,), dtype=torch.long)
    return Data(x=x, edge_index=ei, edge_type=et,
                y=torch.tensor([g['label']], dtype=torch.long),
                nama=g['nama_file'], split=g['split'], folder=g['folder'])

OUT_PYG = f'{D}/data/processed/pyg'
os.makedirs(OUT_PYG, exist_ok=True)

train, bench, kosong = [], [], []
for f in sorted(os.listdir(OUT_PRUNE)):
    if not f.endswith('.json'):
        continue
    with open(f'{OUT_PRUNE}/{f}') as fh:
        g = json.load(fh)
    split = 'train' if g['folder'] in ('0_normal','1_reentrancy') else 'benchmark'
    g['split'] = split
    d = ke_pyg(g)
    if d.num_nodes == 0 or d.edge_index.size(1) == 0:
        kosong.append(g['nama_file'])
        continue
    (train if split=='train' else bench).append(d)

torch.save({'train': train, 'benchmark': bench, 'dim': DIM, 'tipe': TIPE},
           f'{OUT_PYG}/v2_cfg_prune.pt')

import collections
print(f'train    : {len(train)} graf, label={collections.Counter(int(d.y) for d in train)}')
print(f'benchmark: {len(bench)} graf, label={collections.Counter(int(d.y) for d in bench)}')
print(f'dilewati : {len(kosong)}')
print(f'tersimpan: {OUT_PYG}/v2_cfg_prune.pt')

train    : 610 graf, label=Counter({0: 366, 1: 244})
benchmark: 16 graf, label=Counter({0: 8, 1: 8})
dilewati : 1
tersimpan: /content/drive/MyDrive/PROJECT/data/processed/pyg/v2_cfg_prune.pt
